# Part B — Fine-Tuning the Best Model on Food101

This notebook fine-tunes the strongest transfer-learning baseline from Part A by unfreezing deeper backbone blocks and retraining with differential learning rates.

## Notebook Description
- Load the best checkpoint from Part A as the starting point.
- Run controlled fine-tuning experiments with different unfreezing depths.
- Compare validation and test outcomes to measure whether fine-tuning improves over the frozen-backbone baseline.

## Quick Summary
Part B extends the baseline established in Part A: instead of training only the classifier head, selected backbone layers are unfrozen (Experiment 1: 2 layers, Experiment 2: 4 layers, Experiment 3: 6 layers) to test whether deeper adaptation improves classification performance on Food101.

## 1 · Imports & Configuration


In [45]:
# Load shared utilities
try:
    import google.colab  # type: ignore
    get_ipython().run_line_magic("run", "/content/drive/MyDrive/UTS/Semestre 2/Deep Learning/Assignment_2/utils.ipynb")
except Exception:
    get_ipython().run_line_magic("run", "./utils.ipynb")

# ── Reproducibility & Device ───────────────────────────────────────────────────
SEED = 42
setup_utils = SetupUtils()
setup_utils.set_global_seed(SEED)

DEVICE = setup_utils.get_device()
print(f"Using device: {DEVICE}")


Using device: mps


In [46]:
# Shared visualization class instance
visualizer = TrainingVisualizer()
print("TrainingVisualizer ready.")


TrainingVisualizer ready.


## 2 · Path & Hyperparameter Configuration

Set `BEST_MODEL_NAME` to the model that achieved the highest accuracy in Part A.
The corresponding checkpoint will be loaded from `CHECKPOINT_DIR`.

### Fine-tuning knobs

| Variable | Description |
|---|---|
| `BEST_MODEL_NAME` | `"googlenet"`, `"mobilenet"`, or `"resnet"` |
| `NUM_LAYERS_TO_UNFREEZE` | Number of deepest backbone blocks to unfreeze (counting from the end). For MobileNet this uses blocks inside `model.features`; for ResNet/GoogLeNet it uses architecture-aware backbone blocks. Set to `0` to train head only. |
| `BACKBONE_LR` | Learning rate for unfrozen backbone params (typically 10× smaller than head LR) |
| `HEAD_LR` | Learning rate for the classification head |
| `NUM_EPOCHS_FT` | Maximum fine-tuning epochs |


In [ ]:
if all(name in globals() for name in ["full_eval_ds", "train_loader", "val_loader", "test_loader"]):
    classes = full_eval_ds.classes

    visualizer.plot_class_distribution(
        datasets_dict={
            "Train": train_loader.dataset,
            "Val": val_loader.dataset,
            "Test": test_loader.dataset,
        },
        class_names=classes,
    )
else:
    print("Run the Data Loading cell first to plot class distribution.")


In [47]:
# ── Environment ──────────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path(os.path.abspath(""))  # notebook directory

# Checkpoints live in Google Drive when running in Colab so they survive
# session resets. Locally they sit next to the notebook.
GDRIVE_ZIP_PATH = Path("/content/drive/MyDrive/UTS/Semestre 2/Deep Learning/Assignment_2/food-101.zip")
GDRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/UTS/Semestre 2/Deep Learning/Assignment_2/checkpoints")

if IN_COLAB:
    CHECKPOINT_DIR = GDRIVE_CHECKPOINT_DIR
else:
    CHECKPOINT_DIR = BASE_DIR / "checkpoints"

DATA_DIR       = BASE_DIR / "data" / "food-101"
IMG_DIR        = DATA_DIR / "images"
META_DIR       = DATA_DIR / "meta"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Running in Colab : {IN_COLAB}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

# Copy and unzip dataset in Colab (same pattern as Part A)
if IN_COLAB and not DATA_DIR.exists():
    LOCAL_ZIP = Path("/content/food-101.zip")

    if GDRIVE_ZIP_PATH.exists():
        (BASE_DIR / "data").mkdir(parents=True, exist_ok=True)

        print(f"Copying zip from Drive to {LOCAL_ZIP}...")
        shutil.copy(GDRIVE_ZIP_PATH, LOCAL_ZIP)

        print(f"Unzipping dataset to {BASE_DIR / 'data'}...")
        with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
            zip_ref.extractall(BASE_DIR / "data")

        LOCAL_ZIP.unlink()
        print("Extraction complete and zip removed.")
    else:
        raise FileNotFoundError(
            f"Zip file not found at {GDRIVE_ZIP_PATH}. "
            "Check your Google Drive dataset path."
        )
elif IN_COLAB:
    print(f"Dataset already available at {DATA_DIR}")

# ── Best model from Part A ────────────────────────────────────────────────────
# Change this to whichever model scored highest in Part A
BEST_MODEL_NAME = "mobilenet"          # "googlenet" | "mobilenet" | "resnet"

CHECKPOINT_PATH = CHECKPOINT_DIR / f"{BEST_MODEL_NAME}_best.pth"

if not CHECKPOINT_PATH.exists():
    available = list(CHECKPOINT_DIR.glob("*.pth"))
    available_str = "\n  ".join(str(p) for p in available) if available else "  (none found)"
    raise FileNotFoundError(
        f"Part A checkpoint not found: {CHECKPOINT_PATH}\n"
        f"Expected location in Colab: {GDRIVE_CHECKPOINT_DIR / f'{BEST_MODEL_NAME}_best.pth'}\n"
        f"Available checkpoints in {CHECKPOINT_DIR}:\n  {available_str}\n"
        "Make sure Part A checkpoints are saved in your Google Drive checkpoints folder."
    )
print(f"Checkpoint: {CHECKPOINT_PATH}")

# ── Fine-tuning hyperparameters ───────────────────────────────────────────────
#
# NUM_LAYERS_TO_UNFREEZE controls how many of the deepest backbone blocks
# are unfrozen (architecture-aware; MobileNet uses model.features blocks).
#   0  → only the classification head trains (replicates Part A behaviour)
#   2  → last 2 backbone blocks + head  (Experiment 1)
#   4  → last 4 backbone blocks + head  (Experiment 2)
#   6  → last 6 backbone blocks + head  (Experiment 3)
#  -1  → unfreeze EVERYTHING (full fine-tune)
#
# Each experiment cell overrides this value — this is just a shared default.
NUM_LAYERS_TO_UNFREEZE = 2          # default; overridden per experiment

BACKBONE_LR    = 1e-4               # LR for unfrozen backbone params
HEAD_LR        = 1e-3               # LR for the classification head
NUM_EPOCHS_FT  = 15                 # Max fine-tuning epochs
BATCH_SIZE     = 64
NUM_WORKERS    = 2
NUM_CLASSES    = 101
IMG_SIZE       = 224
VAL_FRACTION   = 0.1


print(f"Model          : {BEST_MODEL_NAME}")
print(f"Layers unfrozen: {NUM_LAYERS_TO_UNFREEZE}")
print(f"Backbone LR    : {BACKBONE_LR}")
print(f"Head LR        : {HEAD_LR}")


Running in Colab : False
Checkpoint directory: /Users/juansebastianvargastorres/UTS/Semester2/Deep Learning/CNN_Food101/checkpoints
Checkpoint: /Users/juansebastianvargastorres/UTS/Semester2/Deep Learning/CNN_Food101/checkpoints/mobilenet_best.pth
Model          : mobilenet
Layers unfrozen: 2
Backbone LR    : 0.0001
Head LR        : 0.001


## 3 · Data Loading

This section now mirrors Part A exactly:
- official Food101 train/test splits from metadata files,
- shuffle the official train indices with the same seeded strategy,
- carve validation from that shuffled train set using `VAL_FRACTION`,
- use the same train/eval transforms as Part A.


In [48]:
data_utils = DataPipelineUtils(img_size=IMG_SIZE)
model_utils = ModelUtils(device=DEVICE, num_classes=NUM_CLASSES)
training_utils = TrainingUtils(
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    default_epochs=NUM_EPOCHS_FT,
)

train_transform, eval_transform = data_utils.build_transforms()

splits = data_utils.build_food101_splits_and_loaders(
    img_dir=IMG_DIR,
    meta_dir=META_DIR,
    train_transform=train_transform,
    eval_transform=eval_transform,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    val_fraction=VAL_FRACTION,
    seed=SEED,
)

full_train_ds = splits["full_train_dataset"]
full_eval_ds = splits["full_eval_dataset"]

train_indices = splits["train_indices"]
val_indices = splits["val_indices"]
test_indices = splits["test_indices"]

train_loader = splits["train_loader"]
val_loader = splits["val_loader"]
test_loader = splits["test_loader"]

print(f"Train : {len(train_indices):,} images")
print(f"Val   : {len(val_indices):,} images")
print(f"Test  : {len(test_indices):,} images")


Train : 68,175 images
Val   : 7,574 images
Test  : 25,250 images


## 4 · Experiment 1 — Unfreeze Last 2 Backbone Blocks

We start with a moderate unfreezing: the **two deepest** backbone blocks are unfrozen alongside the head.  
This allows the model to adapt its highest-level feature representations while keeping the majority of pre-trained weights frozen.


In [ ]:
EXP1_LAYERS = 2

model_exp1 = model_utils.prepare_experiment_model(BEST_MODEL_NAME, EXP1_LAYERS, CHECKPOINT_PATH)


In [ ]:
print()
history_exp1, best_acc_exp1 = training_utils.fine_tune(
    model_exp1,
    BEST_MODEL_NAME,
    EXP1_LAYERS,
    train_loader,
    val_loader,
)
print(f"\nExperiment 1 best val accuracy: {best_acc_exp1:.4f}")

# Show train/val curves immediately after Experiment 1 finishes
hist_exp1 = {f"unfreeze={EXP1_LAYERS}": history_exp1}
visualizer.plot_loss(hist_exp1)
visualizer.plot_accuracy(hist_exp1)

# Validation confusion matrix for this experiment
classes = full_eval_ds.classes
visualizer.plot_confusion_matrix(
    model_exp1,
    val_loader,
    classes,
    top_n=20,
    report_top_pairs=True,
    top_k_pairs=1,
)


## 5 · Experiment 2 — Unfreeze Last 4 Backbone Blocks

We now open up more of the backbone to gradient updates.  
With more trainable capacity the model can adapt mid-level features to food-specific patterns, but the risk of **catastrophic forgetting** increases — mitigated by the low `BACKBONE_LR`.


In [ ]:
EXP2_LAYERS = 4

model_exp2 = model_utils.prepare_experiment_model(BEST_MODEL_NAME, EXP2_LAYERS, CHECKPOINT_PATH)


In [ ]:
print()
history_exp2, best_acc_exp2 = training_utils.fine_tune(
    model_exp2,
    BEST_MODEL_NAME,
    EXP2_LAYERS,
    train_loader,
    val_loader,
)
print(f"\nExperiment 2 best val accuracy: {best_acc_exp2:.4f}")

# Show train/val curves immediately after Experiment 2 finishes
hist_exp2 = {f"unfreeze={EXP2_LAYERS}": history_exp2}
visualizer.plot_loss(hist_exp2)
visualizer.plot_accuracy(hist_exp2)

# Validation confusion matrix for this experiment
classes = full_eval_ds.classes
visualizer.plot_confusion_matrix(
    model_exp2,
    val_loader,
    classes,
    top_n=20,
    report_top_pairs=True,
    top_k_pairs=1,
)


## 6 · Experiment 3 — Unfreeze Last 6 Backbone Blocks

This setting unfreezes even deeper backbone capacity to maximize adaptation to Food101.
The tradeoff is higher overfitting risk, so monitor the validation trend closely.


In [ ]:
EXP3_LAYERS = 6

model_exp3 = model_utils.prepare_experiment_model(BEST_MODEL_NAME, EXP3_LAYERS, CHECKPOINT_PATH)

print()
history_exp3, best_acc_exp3 = training_utils.fine_tune(
    model_exp3,
    BEST_MODEL_NAME,
    EXP3_LAYERS,
    train_loader,
    val_loader,
)
print(f"\nExperiment 3 best val accuracy: {best_acc_exp3:.4f}")

# Show train/val curves immediately after Experiment 3 finishes
hist_exp3 = {f"unfreeze={EXP3_LAYERS}": history_exp3}
visualizer.plot_loss(hist_exp3)
visualizer.plot_accuracy(hist_exp3)

# Validation confusion matrix for this experiment
classes = full_eval_ds.classes
visualizer.plot_confusion_matrix(
    model_exp3,
    val_loader,
    classes,
    top_n=20,
    report_top_pairs=True,
    top_k_pairs=1,
)


## 7 · Test Set Evaluation (Best Experiment Only)

Evaluate only the current best experiment on the held-out test set.  
For now, we set **Experiment 3 (unfreeze 6 layers)** as the best candidate.


In [ ]:
print("=" * 55)
print("Test Set Result (Best Experiment)")
print("=" * 55)

BEST_EXP_LAYERS = EXP3_LAYERS  # temporary choice: last experiment
acc_best = training_utils.evaluate_on_test(
    model_utils=model_utils,
    model_name=BEST_MODEL_NAME,
    n_unfrozen=BEST_EXP_LAYERS,
    test_loader=test_loader,
)

# Keep compatibility with summary/comparison cells
acc_exp3 = acc_best


## 8 · Comparison Summary

A concise table comparing the three fine-tuning experiments.


In [ ]:
import pandas as pd

results = pd.DataFrame([
    {
        "Experiment": f"Exp 1 — unfreeze {EXP1_LAYERS} layer(s)",
        "Layers Unfrozen": EXP1_LAYERS,
        "Best Val Acc": f"{best_acc_exp1:.4f}",
        "Test Acc": "-",
    },
    {
        "Experiment": f"Exp 2 — unfreeze {EXP2_LAYERS} layer(s)",
        "Layers Unfrozen": EXP2_LAYERS,
        "Best Val Acc": f"{best_acc_exp2:.4f}",
        "Test Acc": "-",
    },
    {
        "Experiment": f"Exp 3 — unfreeze {EXP3_LAYERS} layer(s) [current best]",
        "Layers Unfrozen": EXP3_LAYERS,
        "Best Val Acc": f"{best_acc_exp3:.4f}",
        "Test Acc": f"{acc_best:.4f}",
    },
])

print(results.to_string(index=False))

                 Experiment  Layers Unfrozen Best Val Acc Test Acc
Exp 1 — unfreeze 2 layer(s)                2       0.6627   0.7111
Exp 2 — unfreeze 3 layer(s)                3       0.6884   0.7354
